In [ ]:
# measure_reflectance.ipynb

# Cell 01 - 

import os
import sys
import time

import numpy as np
import serial

ser = None


def send_cmd(cmd: str) -> str:
    # Append newline character to terminate command
    # Then convert string to bytes() array which is what pySerial expects
    cmd_with_newline = bytes(cmd, "ASCII") + bytes("\n", "ASCII")
    # Send command (Hoyes Modem AT format) to device
    ser.write(cmd_with_newline)
    # Read status line from device, convert bytes() to ASCII, and remove whitespace
    status = ser.readline().decode("ASCII").strip()
    return status


def save_reflectance_data(file_name):
    global ser
    print(
        "Measuring reflectance amplitudes "
        "at 18 different wavelengths over 10 runs..."
    )

    # Connect to the SparkFun Triad Spectroscopy Sensor
    # See https://www.sparkfun.com/products/15050
    port = "COM6"
    if sys.platform == "linux":
        port = "/dev/ttyUSB0"
    if sys.platform == "darwin":
        port = "/dev/tty.usbserial-110"
    ser = serial.Serial(port, 115200, 8, "N", 1, timeout=120)
    time.sleep(2)  # Wait two seconds for sensor to wake up

    # Configure the sensor per the product's datasheet
    # See https://cdn.sparkfun.com/assets/learn_tutorials/8/3/0/AS7265x_Datasheet.pdf
    send_cmd("ATINTTIME=35")  # Set 100ms integration time
    send_cmd("ATGAIN=2")  # Set gain at 16x

    # Set each of the three LEDs (WHT, IR, NIR) on the sesnor
    # to use a 4mA indicator current and 500 mA driver current
    send_cmd("ATLEDC=0x22")  # WHT (AS72651 sensor)
    send_cmd("ATLEDD=0x22")  # IR  (AS72652 sensor)
    send_cmd("ATLEDE=0x22")  # NIR LED (AS7265 sensor)

    # Turn the blue indicator LED off on the sensor
    # and turn on the other three indicator LEDs
    send_cmd("ATLED0=0")  # Turn off blue indicator
    send_cmd("ATLED1=1")  # Turn on WHT LED
    send_cmd("ATLED2=1")  # Turn on IR LED
    send_cmd("ATLED3=1")  # Turn on NIR LED
    time.sleep(2)  # Wait two seconds sensor to warm up

    # Create list of 18 floats to hold the sum of each wavelength levels
    total_readings = [0.0] * 18

    # Perform 10 runs of the experiment
    for n in range(1, 11):
        print(f"Run #{n}: ", end="")
        status = send_cmd("ATCDATA")  # Read all 18 wavelength levels
        # All sucessful commands return a string with "OK" at the end
        print(status)
        # Remove the "OK" then split the CSV string into a list of 18 floats
        readings = [float(s) for s in status.replace("OK", "").split(",")]
        # Accumulate the readings for each wavelength over each run
        for i in range(18):
            total_readings[i] += readings[i]
        # Sleep for two seconds between each run to let sensors settle
        time.sleep(2)

    # Return Sparkfun Triad Sensor to initial condition
    send_cmd("ATLED1=0")  # Turn on WHT LED
    send_cmd("ATLED2=0")  # Turn on IR LED
    send_cmd("ATLED3=0")  # Turn on NIR LED
    send_cmd("ATLED0=1")  # Turn off blue indicator

    # Close the serial port connection to the sensor
    ser.close()

    # Calculate the average of each frequency level over 10 runs
    readings = np.array(total_readings) / 10

    # Reorder levels by increasing wavelength (for the histogram)
    readings = readings[[12, 13, 14, 15, 16, 17, 6, 7, 0, 8, 1, 9, 2, 3, 4, 5, 10, 11]]

    # Determine the path to save the datafile
    folder_path = os.path.dirname(os.path.realpath("__file__"))
    file_path = os.path.join(folder_path, file_name)

    # Use numpy to save the measurements to a CSV file
    np.savetxt(file_path, np.asarray(readings), delimiter=",")
    print(f'Saved readings to "{file_path}"')


datafile_name = "UnknownSample.csv"
save_reflectance_data(datafile_name)

# Plot a reflectance spectra histogram
%run ./plot_reflectance.ipynb